In [ ]:
# Cell 1: Install Dependencies & Mount Drive
import os
import sys

# 1. Install Unsloth and core SFT dependencies cleanly
print("Installing Unsloth and SFT training stack...")
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes transformers datasets

# 2. Mount Google Drive to preserve checkpoints and master weights
from google.colab import drive
drive.mount('/content/drive')

print("✅ Environment successfully initialized.")

In [ ]:
# Cell 2: Global Pipeline Directory Configurations
import os

# Base directory for tracking progress
BASE_DIR = "/content/drive/MyDrive/SFT/modelS/mistral_dsa_buddy"

# Checkpoint path for resuming mid-session runs safely
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")

# Folder to keep your master 16-bit unmerged reference models
OUTPUT_DIR = os.path.join(BASE_DIR, "final_model_16bit")

# Ensure necessary structures exist in Drive before training begins
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Checkpoints Target:  {CHECKPOINT_DIR}")
print(f"📁 Master Model Target: {OUTPUT_DIR}")

In [ ]:
# Cell 2.4: Auto-Patch Checkpoint Configuration
import json
import os

print("🔧 Running Pre-Flight Check on Checkpoint Configurations...")

# 1. Find the latest checkpoint
if os.path.exists(CHECKPOINT_DIR) and os.listdir(CHECKPOINT_DIR):
    checkpoints = [os.path.join(CHECKPOINT_DIR, d) for d in os.listdir(CHECKPOINT_DIR) if "checkpoint-" in d]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        config_path = os.path.join(last_checkpoint, "adapter_config.json")

        if os.path.exists(config_path):
            # 2. Read the config
            with open(config_path, "r") as f:
                config = json.load(f)

            # 3. Force the base model path to the cloud (ignoring local deleted folders)
            target_base = "unsloth/mistral-7b-v0.3-bnb-4bit"
            if config.get("base_model_name_or_path") != target_base:
                print(f"   Fixing broken base path in {last_checkpoint.split('/')[-1]}...")
                config["base_model_name_or_path"] = target_base

                # 4. Save it back
                with open(config_path, "w") as f:
                    json.dump(config, f, indent=4)
                print("✅ Patch applied! Checkpoint is securely anchored to the cloud base model.")
            else:
                print("✅ Configuration is already healthy. No patch needed.")
else:
    print("📝 No checkpoints found to patch yet.")

In [ ]:
# Cell 2.5: Safe 16-Bit Checkpoint Merger & Drive Sync (Network-Immune)
from unsloth import FastLanguageModel
import os
import shutil

# 1. Check if a valid checkpoint exists to merge
last_checkpoint = None
if os.path.exists(CHECKPOINT_DIR) and os.listdir(CHECKPOINT_DIR):
    checkpoints = [os.path.join(CHECKPOINT_DIR, d) for d in os.listdir(CHECKPOINT_DIR) if "checkpoint-" in d]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"🔄 Found target checkpoint for compilation: {last_checkpoint}")

if not last_checkpoint:
    print("📝 Notice: No checkpoints found in Drive yet. Skipping compilation block for now.")
else:
    # 2. Define local and cloud paths
    local_temp_path = "/content/pristine_model_16bit"

    print("\n📦 Loading model structure directly from your active checkpoint...")
    compile_model, compile_tokenizer = FastLanguageModel.from_pretrained(
        model_name = last_checkpoint,
        max_seq_length = 2560,
        load_in_4bit = True  # Keeps it lightweight during RAM loading
    )

    print("\n⏳ Step 1: Writing 16-bit weights to fast local disk (bypassing Google Drive network lag)...")
    if os.path.exists(local_temp_path):
        shutil.rmtree(local_temp_path)

    # Compile matrices locally where write speeds are steady and handles won't drop
    compile_model.save_pretrained_merged(local_temp_path, compile_tokenizer, save_method = "merged_16bit")
    print("✅ Local compilation complete! Verifying file sizes...")

    # 3. Size Verification Loop
    all_shards_healthy = True
    for f in sorted(os.listdir(local_temp_path)):
        file_bytes = os.path.getsize(os.path.join(local_temp_path, f))
        size_gb = file_bytes / (1024**3)
        print(f"   📦 File: {f} | Size: {size_gb:.2f} GB")
        if f.endswith(".safetensors") and file_bytes == 0:
            all_shards_healthy = False

    if not all_shards_healthy:
        print("❌ Error: A local file size registered as 0 bytes. Compiling failed.")
    else:
        # 4. Clean Block Transfer to Google Drive
        print(f"\n🔄 Step 2: Preparing Google Drive target directory at: {OUTPUT_DIR}")
        if os.path.exists(OUTPUT_DIR):
            print("   Removing old folder containing 0-byte ghost files...")
            shutil.rmtree(OUTPUT_DIR)

        print("🚀 Step 3: Copying pristine local shards sequentially to Google Drive. Please wait...")
        # shutil.copytree passes the files as a predictable bitstream that Drive won't drop
        shutil.copytree(local_temp_path, OUTPUT_DIR)

        print(f"\n🎉 SUCCESS! All 3 shards have safely landed with real file sizes at:\n👉 {OUTPUT_DIR}")

    # Clean up local cache space immediately to keep the notebook light
    shutil.rmtree(local_temp_path)

    # Delete the temporary variables to free up GPU memory for the main training pipeline
    del compile_model, compile_tokenizer
    import torch; torch.cuda.empty_cache()

In [ ]:
# Cell 3: Synchronized Checkpoint Tracking & Stream Slicing
import os
from datasets import load_dataset, Dataset

# 1. Calculate exactly how many rows the model has historically trained on
rows_already_trained = 0

if os.path.exists(CHECKPOINT_DIR) and os.listdir(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if "checkpoint-" in d]
    if checkpoints:
        latest_step = max([int(d.split("-")[-1]) for d in checkpoints])
        # 1 Step = 8 Rows (per_device_train_batch_size=1 * gradient_accumulation_steps=8)
        rows_already_trained = latest_step * 8
        print(f"🔄 Resuming Dataset: Found model checkpoint at Step {latest_step}.")
        print(f"⏩ Skipping exactly {rows_already_trained} rows to match the model's exact state.")
else:
    print("📝 Resuming Dataset: No checkpoints found. Starting fresh from row 0.")

print("📦 Loading nvidia/OpenCodeReasoning from Hugging Face in streaming mode...")
# Explicitly extraction of the subset out of the dictionary map to fix the stream bug
raw_dataset = load_dataset("nvidia/OpenCodeReasoning", "split_0", streaming=True)["split_0"]

# 2. Fast-forward the data stream to match the model's exact row location
if rows_already_trained > 0:
    raw_dataset = raw_dataset.skip(rows_already_trained)

# 3. Create an active iterable stream to pull rows safely into memory
dataset_iter = iter(raw_dataset)

# Collect exactly 1000 items for the daily local slice
samples = []
print("⏳ Extracting rows from stream (this may take a moment)...")

while len(samples) < 1000:
    try:
        item = next(dataset_iter)
    except StopIteration:
        print("⚠️ Reached the absolute end of the dataset stream!")
        break
    except Exception as e:
        continue

    # Map target keys dynamically based on dataset configuration
    prompt_text = item.get("instruction", item.get("input", ""))
    response_text = item.get("output", item.get("solution", item.get("response", "")))

    if not prompt_text or not response_text:
        continue

    samples.append({
        "instruction": prompt_text,
        "full_response": response_text
    })

    if len(samples) % 200 == 0:
        print(f"   Collected {len(samples)}/1000 rows...")

if len(samples) == 0:
    raise ValueError("❌ Error: The extracted samples list is empty! Check stream configurations.")

# 4. Split into your pristine 800/200 train/val layout
full_daily_dataset = Dataset.from_list(samples)
split_dataset = full_daily_dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"📊 Ready for this session: {len(train_dataset)} Train rows | {len(val_dataset)} Validation rows.")

In [ ]:
# Cell 4: Prompt Preprocessing Function for NVIDIA Reasoning Schema
def format_dsa_prompt(example):
    problem = example['instruction']
    full_response = example['full_response'] # Contains <thought> segments natively

    augmented_instruction = f"{problem}\n\nPlease provide your reasoning and the solution strictly in Python."

    # Structure seamlessly into Mistral's instruction block format
    text = (
        f"<s>[INST] You are an expert DSA Interview Buddy. Analyze the problem, map your constraints "
        f"inside <thought>...</thought> tags, and then provide a friendly explanation with clean Python code.\n\n"
        f"Problem: {augmented_instruction} [/INST]\n{full_response}</s>"
    )
    return {"text": text}

train_dataset = train_dataset.map(format_dsa_prompt)
val_dataset = val_dataset.map(format_dsa_prompt)
print("✨ Mapping complete. Payload strings are aligned with structural template boundaries.")

In [ ]:
# Cell 5: Load Base Model with Dynamic Local Fallbacks
from unsloth import FastLanguageModel

max_seq_length = 2560

# Look at your target master directory to determine if you are booting up for the first time
if os.path.exists(OUTPUT_DIR) and any(f.endswith('.safetensors') for f in os.listdir(OUTPUT_DIR)):
    model_source = OUTPUT_DIR
    print(f"📦 Loading your custom fine-tuned 16-bit model from Google Drive...")
else:
    model_source = "unsloth/mistral-7b-v0.3-bnb-4bit"
    print(f"🌐 No local master weights found. Initializing from Hugging Face hub standard baseline...")

# On-the-fly 4-bit quantization keeps standard T4 attention steps comfortably below VRAM ceilings
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_source,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Set up parameter-efficient fine-tuning matrices
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial for keeping peak activation memory minimal
    random_state = 3407,
)
print(f"🚀 Model initialized with optimized parameters and a {max_seq_length} token context window.")

In [ ]:
# Cell 6: Run SFT Training (Dynamic Multi-Session Support)
import os
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# Prevent memory fragmentation issues on older architecture GPUs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. Detect last save step checkpoint dynamically
last_checkpoint = None
current_step = 0

if os.path.exists(CHECKPOINT_DIR) and os.listdir(CHECKPOINT_DIR):
    checkpoints = [os.path.join(CHECKPOINT_DIR, d) for d in os.listdir(CHECKPOINT_DIR) if "checkpoint-" in d]
    if checkpoints:
        last_checkpoint = max(checkpoints, key=os.path.getmtime)
        current_step = int(last_checkpoint.split("-")[-1])
        print(f"🔄 Found valid checkpoint at Step {current_step}. Resuming training execution path...")

# 2. Append exactly 100 incremental steps to the tracked baseline
target_max_steps = current_step + 100
print(f"🎯 Target boundary for this session: Step {target_max_steps}")

# 3. Initialize Trainer Configuration
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # Minimum base step footprint
        gradient_accumulation_steps = 8,  # Maintains global effective batch size of 8
        warmup_steps = 5,
        max_steps = target_max_steps,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "paged_adamw_8bit",       # Pages optimizer states out to CPU RAM to dodge OOMs
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,
        output_dir = CHECKPOINT_DIR,
        report_to = "none"
    ),
)

# 4. Fire the calculation loop
trainer.train(resume_from_checkpoint = last_checkpoint)